In [1]:
import scala.math
import org.apache.spark._
import org.apache.spark.sql._
import org.apache.spark.rdd._
import org.apache.spark.ml.linalg._
import org.apache.spark.sql.functions._
import org.apache.spark.sql.functions.{col}
import org.apache.spark.sql.{DataFrame, Row, SparkSession}
import org.apache.spark.ml.{Pipeline, PipelineModel, PipelineStage}
import org.apache.spark.ml.tuning.{ ParamGridBuilder, CrossValidator}
import org.apache.spark.ml.classification.{MultilayerPerceptronClassifier, RandomForestClassifier, GBTClassifier}
import org.apache.spark.ml.feature.{StandardScaler, VectorAssembler, StringIndexer, MinMaxScaler}
import org.apache.spark.ml.evaluation.{MulticlassClassificationEvaluator,BinaryClassificationEvaluator}
import org.apache.spark.ml.clustering._

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
587,application_1580996944851_0543,spark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

import scala.math
import org.apache.spark._
import org.apache.spark.sql._
import org.apache.spark.rdd._
import org.apache.spark.ml.linalg._
import org.apache.spark.sql.functions._
import org.apache.spark.sql.functions.col
import org.apache.spark.sql.{DataFrame, Row, SparkSession}
import org.apache.spark.ml.{Pipeline, PipelineModel, PipelineStage}
import org.apache.spark.ml.tuning.{ParamGridBuilder, CrossValidator}
import org.apache.spark.ml.classification.{MultilayerPerceptronClassifier, RandomForestClassifier, GBTClassifier}
import org.apache.spark.ml.feature.{StandardScaler, VectorAssembler, StringIndexer, MinMaxScaler}
import org.apache.spark.ml.evaluation.{MulticlassClassificationEvaluator, BinaryClassificationEvaluator}
import org.apache.spark.ml.clustering._


In [2]:
object osbnrClass{

    def Osbnr(data: DataFrame, k: Int) : DataFrame = {

        val Majority = data.filter($"Class" === "0")
        val Minority = data.filter($"Class" === "1")
        val modelKmeans = new KMeans().setK(k).setFeaturesCol("features").setPredictionCol("prediction").setSeed(1L).setMaxIter(100)
        val modelk = modelKmeans.fit(Minority)
        var predictionsMin = modelk.transform(Minority)
        val centroids=modelk.clusterCenters
        var predictionsMaj = modelk.transform(Majority)
        val max = maxByCentroid(centroids, predictionsMin, k)
        val noisydata = getNoisyInstances(centroids, max, predictionsMaj).drop("dist","prediction","noisy").checkpoint()
        val newMajority = Majority.exceptAll(noisydata)
        val newData = newMajority.union(Minority).sort("Time")
        
    return newData
}
    
def distance(centroid: Vector, data: Vector): Double = math.sqrt(centroid.toArray.zip(data.toArray).map(p => p._1 - p._2).map(d => d * d).sum)
def distanceAllCluster(centroid: Vector, dataCentroid: Array[DenseVector]): Array[Double] = {
      dataCentroid.map(d => distance(centroid, d))
    }
//DenseVector
def maxByCentroid(centroids: Array[Vector], data: DataFrame, k: Int): Map[Int, Double] = {
      val max = (0 until k).map{ k => val dataCentroid = data.filter($"prediction" === k).select("features").collect().map {
            case Row(v: Vector) => v.toDense
          }
        val dist = distanceAllCluster(centroids(k), dataCentroid)
        if (dist.isEmpty) {
          (k, 0.0)
        }
        else
          (k, dist.max)
      }
      max.toMap
    }
def calculateDistance(centroids: Array[Vector]) = udf((v: Vector, k: Int) => {
      math.sqrt(centroids(k).toArray.zip(v.toArray)
        .map(p => p._1 - p._2).map(d => d * d).sum)
    })
    
def checkNoisy(max: Map[Int, Double]) = udf((distance: Double, k: Int) => if (distance <= max(k)) 1 else 0)
    
def getNoisyInstances(centroids: Array[Vector], max: Map[Int, Double], PredictionMaj: DataFrame) = {
    val distanceDF = PredictionMaj.withColumn("dist", calculateDistance(centroids)(PredictionMaj("features"), PredictionMaj("prediction"))).checkpoint()
      val noiseinstances = distanceDF.withColumn("noisy", checkNoisy(max)(distanceDF("dist"), distanceDF("prediction"))).checkpoint()
        noiseinstances.filter($"noisy" > 0).dropDuplicates
    }
}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

defined object osbnrClass


In [3]:
sc.setCheckpointDir("checkpoints/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
val raw = spark.read.format("csv").option("header", "true").option("mode", "DROPMALFORMED").csv("datasets/creditcard.csv")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

raw: org.apache.spark.sql.DataFrame = [Time: string, V1: string ... 29 more fields]


In [5]:
// cast all the column to Double type.
val df = raw.select(((1 to 28).map(i => "V" + i) ++ Array("Time", "Amount", "Class")).map(s => col(s).cast("Double")): _*)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

df: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 29 more fields]


In [6]:
// Preparing Data 
val labelConverter = new StringIndexer().setInputCol("Class").setOutputCol("label")
val assembler = new VectorAssembler().setInputCols(Array( "V3", "V4", "V9", "V10", "V11", "V12", "V14", "V16", "V17")).setOutputCol("assembled")
val scaler = new StandardScaler().setInputCol("assembled").setOutputCol("features")
val pipeline = new Pipeline().setStages(Array(assembler, scaler, labelConverter))
val pipelineModel = pipeline.fit(df)
val data = pipelineModel.transform(df)
println("Generate feature from raw data:")
data.select("features","label").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

labelConverter: org.apache.spark.ml.feature.StringIndexer = strIdx_17601042931e
assembler: org.apache.spark.ml.feature.VectorAssembler = vecAssembler_dd1afc7c6df3
scaler: org.apache.spark.ml.feature.StandardScaler = stdScal_7fd806566587
pipeline: org.apache.spark.ml.Pipeline = pipeline_c1fbda9ada4e
pipelineModel: org.apache.spark.ml.PipelineModel = pipeline_c1fbda9ada4e
data: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 32 more fields]
Generate feature from raw data:
+--------------------+-----+
|            features|label|
+--------------------+-----+
|[1.67277056254252...|  0.0|
|[0.10979690934882...|  0.0|
|[1.16946643973207...|  0.0|
|[1.18251437486170...|  0.0|
|[1.02140988238942...|  0.0|
|[0.75258405639254...|  0.0|
|[0.02992291760621...|  0.0|
|[0.70857499080745...|  0.0|
|[-0.0746524907375...|  0.0|
|[0.68878028300438...|  0.0|
|[0.60270853498396...|  0.0|
|[-0.5766178509613...|  0.0|
|[0.25320948651198...|  0.0|
|[0.54648639167204...|  0.0|
|[1.08276652341442.

In [7]:
val data0 = data.filter($"Class" === "0").cache()
val data1 = data.filter($"Class" === "1").cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

data0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
data1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]


In [8]:
val splitTime0 = data0.stat.approxQuantile("Time", Array(0.7), 0).head
val splitTime1 = data1.stat.approxQuantile("Time", Array(0.8), 0).head
val trainingData0 = data0.filter(s"Time<$splitTime0").cache()
val validationData0 = data.filter(s"Time>=$splitTime0").cache()
val trainingData1 = data1.filter(s"Time<$splitTime1").cache()
val validationData1 = data1.filter(s"Time>=$splitTime1").cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

splitTime0: Double = 132946.0
splitTime1: Double = 137211.0
trainingData0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
validationData0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
trainingData1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
validationData1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]


In [9]:
val trainingData = trainingData0.unionAll(trainingData1)
val validationData = validationData0.unionAll(validationData1)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

trainingData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]
validationData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [V1: double, V2: double ... 32 more fields]


In [10]:
println(" Training set statistics: 1 represents fraud and 0 represents normal")
trainingData.groupBy("Class").count().show()
println(" Test set statistics: 1 represents fraud and 0 represents normal")
validationData.groupBy("Class").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

 Training set statistics: 1 represents fraud and 0 represents normal
+-----+------+
|Class| count|
+-----+------+
|  0.0|199017|
|  1.0|   393|
+-----+------+

 Test set statistics: 1 represents fraud and 0 represents normal
+-----+-----+
|Class|count|
+-----+-----+
|  0.0|85298|
|  1.0|  207|
+-----+-----+



# Under-Sampling OSBNR

In [11]:
val s = System.nanoTime
val newtrainingdata = osbnrClass.Osbnr(trainingData, 2)
val durationresampling = (System.nanoTime - s) / 1e9d
println(s"Resampling process takes $durationresampling secs")
println("New training set statistics: 1 represents Minority class and 0 represents Majority class")
newtrainingdata.groupBy("Class").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s: Long = 31481665088277861
newtrainingdata: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 32 more fields]
durationresampling: Double = 27.045339746
Resampling process takes 27.045339746 secs
New training set statistics: 1 represents Minority class and 0 represents Majority class
+-----+-----+
|Class|count|
+-----+-----+
|  0.0| 1944|
|  1.0|  393|
+-----+-----+



# Learning Model 

In [12]:
// Create a RandomForest model.
val RF = new RandomForestClassifier().setLabelCol("label").setFeaturesCol("features").setNumTrees(20).setImpurity("gini").setMaxBins(32).setMaxDepth(8).setFeatureSubsetStrategy("auto")
val t = System.nanoTime

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RF: org.apache.spark.ml.classification.RandomForestClassifier = rfc_dbe31e68aedd
t: Long = 31481707081600949


In [13]:
// train the RandomForest model.
val Modelrf = RF.fit(newtrainingdata)
val durationtrain = (System.nanoTime - t) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationtrain secs")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Modelrf: org.apache.spark.ml.classification.RandomForestClassificationModel = RandomForestClassificationModel (uid=rfc_dbe31e68aedd) with 20 trees
durationtrain: Double = 30.448101596

initial model training finished.
Training process takes 30.448101596 secs


In [14]:
val s = System.nanoTime
val predictionsRF = Modelrf.transform(validationData)
val durationprediction = (System.nanoTime - s) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationprediction secs")
predictionsRF.select("prediction", "label", "features").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s: Long = 31481739211772120
predictionsRF: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 35 more fields]
durationprediction: Double = 0.296858376

initial model training finished.
Training process takes 0.296858376 secs
+----------+-----+--------------------+
|prediction|label|            features|
+----------+-----+--------------------+
|       0.0|  0.0|[0.05081706126562...|
|       0.0|  0.0|[-0.7683039709008...|
|       0.0|  0.0|[0.18913098116961...|
|       0.0|  0.0|[-4.5819156007642...|
|       0.0|  0.0|[-1.3072335549455...|
+----------+-----+--------------------+
only showing top 5 rows



In [15]:
println(s"Matrice de confusion :")
predictionsRF.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Matrice de confusion :
+----------+-----+-----+
|prediction|label|count|
+----------+-----+-----+
|       0.0|  0.0|84077|
|       1.0|  0.0| 1221|
|       0.0|  1.0|   35|
|       1.0|  1.0|  172|
+----------+-----+-----+



In [16]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsRF)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsRF))
println("Accuracy = " + evaluator1.evaluate(predictionsRF))
println("Precision = " + evaluator2.evaluate(predictionsRF))
println("Recall = " + evaluator3.evaluate(predictionsRF))
println("F1 = " + evaluator4.evaluate(predictionsRF))
println("Test Error = " + (1.0 - accuracy))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

evaluator1: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_550e21b7f640
evaluator2: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_6b2cdf27df86
evaluator3: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_d3b75aedae29
evaluator4: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_da0040fbacb8
areaUnderROC: org.apache.spark.ml.evaluation.BinaryClassificationEvaluator = binEval_be466deea203
accuracy: Double = 0.9853108005379803
Area Under ROC Curve = 0.9083016767699217
Accuracy = 0.9853108005379803
Precision = 0.9974629052808581
Recall = 0.9853108005379801
F1 = 0.9907035672544554
Test Error = 0.014689199462019742


In [17]:
val rfModel = {
    val rfGridSearch = for (
    rfNumTrees <- Array(10, 15, 20);
    rfImpurity <- Array("entropy","gini");
    rfMaxBins <- Array(24, 28, 32);
    rfmaxDepth <- Array(4, 6, 8))
   // rfsubsample <- Array(0.25,0.45,0.65)) 
    yield {
   println(s"Training random forest numTrees : $rfNumTrees, impurity : $rfImpurity, maxBins: $rfMaxBins, maxDepth : $rfmaxDepth")     
   val rfModel = new RandomForestClassifier().setLabelCol("label").setFeaturesCol("features").setNumTrees(rfNumTrees).setImpurity(rfImpurity).setMaxDepth(rfmaxDepth).setMaxBins(rfMaxBins).setFeatureSubsetStrategy("auto").fit(newtrainingdata)
   val predictionsRF = rfModel.transform(validationData)      
   val rfAUC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC").evaluate(predictionsRF)  
   println("Area Under ROC Curve = " + rfAUC)
       ((rfNumTrees, rfImpurity, rfMaxBins, rfmaxDepth), rfModel, rfAUC)
    }  
    println(rfGridSearch.sortBy(-_._3).take(5).mkString("\n"))
        val BestModelRF = rfGridSearch.sortBy(-_._3).head._2
    BestModelRF
}   

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Training random forest numTrees : 10, impurity : entropy, maxBins: 24, maxDepth : 4
Area Under ROC Curve = 0.871898611098368
Training random forest numTrees : 10, impurity : entropy, maxBins: 24, maxDepth : 6
Area Under ROC Curve = 0.8999419539997483
Training random forest numTrees : 10, impurity : entropy, maxBins: 24, maxDepth : 8
Area Under ROC Curve = 0.9109996915615988
Training random forest numTrees : 10, impurity : entropy, maxBins: 28, maxDepth : 4
Area Under ROC Curve = 0.8719337819112828
Training random forest numTrees : 10, impurity : entropy, maxBins: 28, maxDepth : 6
Area Under ROC Curve = 0.9122818404314378
Training random forest numTrees : 10, impurity : entropy, maxBins: 28, maxDepth : 8
Area Under ROC Curve = 0.9132627436428331
Training random forest numTrees : 10, impurity : entropy, maxBins: 32, maxDepth : 4
Area Under ROC Curve = 0.871910334702673
Training random forest numTrees : 10, impurity : entropy, maxBins: 32, maxDepth : 6
Area Under ROC Curve = 0.90345238058

In [18]:
val predictionsBestRF = rfModel.transform(validationData)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

predictionsBestRF: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 35 more fields]


In [19]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsBestRF)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsBestRF))
println("Accuracy = " + evaluator1.evaluate(predictionsBestRF))
println("Precision = " + evaluator2.evaluate(predictionsBestRF))
println("Recall = " + evaluator3.evaluate(predictionsBestRF))
println("F1 = " + evaluator4.evaluate(predictionsBestRF))
println("Test Error = " + (1.0 - accuracy))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

evaluator1: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_4bcb36403893
evaluator2: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_895a6846e155
evaluator3: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_7209564fc558
evaluator4: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_5cbefaae3c33
areaUnderROC: org.apache.spark.ml.evaluation.BinaryClassificationEvaluator = binEval_809cb483ca6f
accuracy: Double = 0.9807028828723466
Area Under ROC Curve = 0.922859306667174
Accuracy = 0.9807028828723466
Precision = 0.9974859991575572
Recall = 0.9807028828723466
F1 = 0.98827114332057
Test Error = 0.019297117127653363


In [20]:
println(s"Matrice de confusion :")
   predictionsBestRF.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Matrice de confusion :
+----------+-----+-----+
|prediction|label|count|
+----------+-----+-----+
|       0.0|  0.0|83676|
|       1.0|  0.0| 1622|
|       0.0|  1.0|   28|
|       1.0|  1.0|  179|
+----------+-----+-----+



In [21]:
// create Multilayer Perceptron Model and set its parameters
val t = System.nanoTime
val layers = Array[Int] (9, 30, 15, 2)
//(9, 30, 15, 2)
val mlp = new MultilayerPerceptronClassifier().setLayers(layers).setLabelCol("label").setFeaturesCol("features").setTol(1E-4).setMaxIter(25) 

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

t: Long = 31483059394552263
layers: Array[Int] = Array(9, 30, 15, 2)
mlp: org.apache.spark.ml.classification.MultilayerPerceptronClassifier = mlpc_0b8b3eae6593


In [22]:
// Train a Multilayer Perceptron model.
val modelMLP = mlp.fit(newtrainingdata)
val durationtrain = (System.nanoTime - t) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationtrain secs")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

modelMLP: org.apache.spark.ml.classification.MultilayerPerceptronClassificationModel = mlpc_0b8b3eae6593
durationtrain: Double = 30.128452487

initial model training finished.
Training process takes 30.128452487 secs


In [23]:
val s = System.nanoTime
val predictionsMLP = modelMLP.transform(validationData)
val durationprediction = (System.nanoTime - s) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationprediction secs")
predictionsMLP.select("prediction","label").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s: Long = 31483091737609815
predictionsMLP: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 35 more fields]
durationprediction: Double = 0.266047811

initial model training finished.
Training process takes 0.266047811 secs
+----------+-----+
|prediction|label|
+----------+-----+
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       1.0|  0.0|
+----------+-----+
only showing top 5 rows



In [24]:
println(s"Classified test set :")
validationData.groupBy("Class").count().show()
println(s"Prediction :")
predictionsMLP.groupBy("prediction").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Classified test set :
+-----+-----+
|Class|count|
+-----+-----+
|  0.0|85298|
|  1.0|  207|
+-----+-----+

Prediction :
+----------+-----+
|prediction|count|
+----------+-----+
|       0.0|71003|
|       1.0|14502|
+----------+-----+



In [25]:
println(s"Matrice de confusion :")
predictionsMLP.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Matrice de confusion :
+----------+-----+-----+
|prediction|label|count|
+----------+-----+-----+
|       0.0|  0.0|70979|
|       1.0|  0.0|14319|
|       0.0|  1.0|   24|
|       1.0|  1.0|  183|
+----------+-----+-----+



In [26]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsMLP)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsMLP))
println("Accuracy = " + evaluator1.evaluate(predictionsMLP))
println("Precision = " + evaluator2.evaluate(predictionsMLP))
println("Recall = " + evaluator3.evaluate(predictionsMLP))
println("F1 = " + evaluator4.evaluate(predictionsMLP))
println("Test Error = " + (1.0 - accuracy))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

evaluator1: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_2fc79b8e994c
evaluator2: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_7f7468817352
evaluator3: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_346b4332238f
evaluator4: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_8766a6e750b4
areaUnderROC: org.apache.spark.ml.evaluation.BinaryClassificationEvaluator = binEval_a3b5fa724771
accuracy: Double = 0.8322554236594352
Area Under ROC Curve = 0.8580938404862611
Accuracy = 0.8322554236594352
Precision = 0.9972724427104257
Recall = 0.8322554236594351
F1 = 0.9060962354860046
Test Error = 0.16774457634056483


In [27]:
val mlpModel = {
    val mlpGridSearch = for (
    mlplayers <- Array( Array[Int] (9, 60, 30, 2), Array[Int] (9, 13, 4, 2), Array[Int] (9, 58, 20, 2));
    mlpMaxIter <- Array(25, 50, 100);
    mlpTol <- Array(1E-4, 1E-6)) 
    yield {
   println(s"Training multilayers perception layers : $mlplayers, Tolerance : $mlpTol, Iterations: $mlpMaxIter")    
   val mlpModel = new MultilayerPerceptronClassifier().setLayers(mlplayers).setLabelCol("label").setFeaturesCol("features").setTol(mlpTol).setBlockSize(128).setSeed(1234L).setMaxIter(mlpMaxIter).fit(newtrainingdata) 
   val predictionsMLP = mlpModel.transform(validationData)      
   val mlpAUC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC").evaluate(predictionsMLP)  
   println("Area Under ROC Curve = " + mlpAUC)
            ((mlplayers, mlpMaxIter, mlpTol), mlpModel, mlpAUC)
    }
    
    println(mlpGridSearch.sortBy(-_._3).take(5).mkString("\n"))
        val BestModel = mlpGridSearch.sortBy(-_._3).head._2
    BestModel
}   

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Training multilayers perception layers : [I@5e2b8ac2, Tolerance : 1.0E-4, Iterations: 25
Area Under ROC Curve = 0.8901309679517436
Training multilayers perception layers : [I@5e2b8ac2, Tolerance : 1.0E-6, Iterations: 25
Area Under ROC Curve = 0.8901309679517436
Training multilayers perception layers : [I@5e2b8ac2, Tolerance : 1.0E-4, Iterations: 50
Area Under ROC Curve = 0.8450638472021307
Training multilayers perception layers : [I@5e2b8ac2, Tolerance : 1.0E-6, Iterations: 50
Area Under ROC Curve = 0.8450638472021307
Training multilayers perception layers : [I@5e2b8ac2, Tolerance : 1.0E-4, Iterations: 100
Area Under ROC Curve = 0.8555354045487359
Training multilayers perception layers : [I@5e2b8ac2, Tolerance : 1.0E-6, Iterations: 100
Area Under ROC Curve = 0.8555354045487359
Training multilayers perception layers : [I@6e58ead, Tolerance : 1.0E-4, Iterations: 25
Area Under ROC Curve = 0.8810904832311114
Training multilayers perception layers : [I@6e58ead, Tolerance : 1.0E-6, Iteration

In [28]:
val predictionsBestMLP = mlpModel.transform(validationData)
val durationprediction = (System.nanoTime - s) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationprediction secs")
predictionsBestMLP.select("prediction","label").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

predictionsBestMLP: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 35 more fields]
durationprediction: Double = 888.836619332

initial model training finished.
Training process takes 888.836619332 secs
+----------+-----+
|prediction|label|
+----------+-----+
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       1.0|  0.0|
+----------+-----+
only showing top 5 rows



In [29]:
println(s"Matrice de confusion :")
predictionsBestMLP.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Matrice de confusion :
+----------+-----+-----+
|prediction|label|count|
+----------+-----+-----+
|       0.0|  0.0|73972|
|       1.0|  0.0|11326|
|       0.0|  1.0|   18|
|       1.0|  1.0|  189|
+----------+-----+-----+



In [30]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsBestMLP)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsBestMLP))
println("Accuracy = " + evaluator1.evaluate(predictionsBestMLP))
println("Precision = " + evaluator2.evaluate(predictionsBestMLP))
println("Recall = " + evaluator3.evaluate(predictionsBestMLP))
println("F1 = " + evaluator4.evaluate(predictionsBestMLP))
println("Test Error = " + (1.0 - accuracy))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

evaluator1: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_acc269e349eb
evaluator2: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_753ea733377d
evaluator3: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_2e4e883b3acd
evaluator4: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_c761e41efffc
areaUnderROC: org.apache.spark.ml.evaluation.BinaryClassificationEvaluator = binEval_5fcbffa49a9f
accuracy: Double = 0.8673293959417578
Area Under ROC Curve = 0.8901309679517436
Accuracy = 0.8673293959417578
Precision = 0.9973761370918923
Recall = 0.8673293959417577
F1 = 0.9266126507433204
Test Error = 0.13267060405824216


In [31]:
//Gradient-boosted tree classifier
// Train a GBT model.
//setImpurity("entropy") 
val t = System.nanoTime
val gbt = new GBTClassifier().setLabelCol("label").setFeaturesCol("features").setMaxIter(10)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

t: Long = 31483988167289836
gbt: org.apache.spark.ml.classification.GBTClassifier = gbtc_3cf9fcbd0c35


In [32]:
val modelGBT = gbt.fit(newtrainingdata)
val durationtrain = (System.nanoTime - t) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationtrain secs")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

modelGBT: org.apache.spark.ml.classification.GBTClassificationModel = GBTClassificationModel (uid=gbtc_3cf9fcbd0c35) with 10 trees
durationtrain: Double = 62.42342267

initial model training finished.
Training process takes 62.42342267 secs


In [33]:
val s = System.nanoTime
val predictionsGBT = modelGBT.transform(validationData)
val durationprediction = (System.nanoTime - s) / 1e9d
println("\ninitial model training finished.")
println(s"Training process takes $durationprediction secs")
predictionsGBT.cache()
//val predictionsAndLabel: RDD[Row] = df.rdd= predictionsGBT.select("prediction", "label")
predictionsGBT.select("prediction", "label").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s: Long = 31484051970446449
predictionsGBT: org.apache.spark.sql.DataFrame = [V1: double, V2: double ... 35 more fields]
durationprediction: Double = 0.250732615

initial model training finished.
Training process takes 0.250732615 secs
res74: predictionsGBT.type = [V1: double, V2: double ... 35 more fields]
+----------+-----+
|prediction|label|
+----------+-----+
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
+----------+-----+
only showing top 5 rows



In [34]:
println(s"Classified test set :")
validationData.groupBy("Class").count().show()
println(s"Prediction :")
predictionsGBT.groupBy("prediction").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Classified test set :
+-----+-----+
|Class|count|
+-----+-----+
|  0.0|85298|
|  1.0|  207|
+-----+-----+

Prediction :
+----------+-----+
|prediction|count|
+----------+-----+
|       0.0|83039|
|       1.0| 2466|
+----------+-----+



In [35]:
println(s"Matrice de confusion :")
predictionsGBT.select("prediction", "label").groupBy("prediction", "label").count().orderBy("label", "prediction").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Matrice de confusion :
+----------+-----+-----+
|prediction|label|count|
+----------+-----+-----+
|       0.0|  0.0|83008|
|       1.0|  0.0| 2290|
|       0.0|  1.0|   31|
|       1.0|  1.0|  176|
+----------+-----+-----+



In [36]:
val evaluator1 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("accuracy")
val evaluator2 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedPrecision")
val evaluator3 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("weightedRecall")
val evaluator4 = new MulticlassClassificationEvaluator().setLabelCol("label").setPredictionCol("prediction").setMetricName("f1")
val areaUnderROC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC")
val accuracy = evaluator1.evaluate(predictionsGBT)
println("Area Under ROC Curve = " + areaUnderROC.evaluate(predictionsGBT))
println("Accuracy = " + evaluator1.evaluate(predictionsGBT))
println("Precision = " + evaluator2.evaluate(predictionsGBT))
println("Recall = " + evaluator3.evaluate(predictionsGBT))
println("F1 = " + evaluator4.evaluate(predictionsGBT))
println("Test Error = " + (1.0 - accuracy))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

evaluator1: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_9c66a6f31dc5
evaluator2: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_0d7277e7d6bf
evaluator3: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_9d5d8f68b528
evaluator4: org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator = mcEval_1476c2c39d50
areaUnderROC: org.apache.spark.ml.evaluation.BinaryClassificationEvaluator = binEval_600d8ebd6933
accuracy: Double = 0.9728553885737676
Area Under ROC Curve = 0.9116972460177408
Accuracy = 0.9728553885737676
Precision = 0.9973794561253504
Recall = 0.9728553885737676
F1 = 0.9841434526435006
Test Error = 0.027144611426232368


In [37]:
val GBTModel = {
    val gbtGridSearch = for (
    gbtMaxIter <- Array(10, 15, 20);
    gbtImpurity <- Array("entropy","gini");
    gbtMaxBins <- Array(24, 28, 30);
    gbtmaxDepth <- Array(2, 4, 6);
   gbtsubsample <- Array(0.25,0.45,0.65)) 
    yield {
   println(s"Training GBT MaxIter : $gbtMaxIter, SubsamplingRate : $gbtsubsample, impurity : $gbtImpurity, maxBins: $gbtMaxBins, maxDepth : $gbtmaxDepth")     
   val gbtModel = new GBTClassifier().setLabelCol("label").setFeaturesCol("features").setSubsamplingRate(gbtsubsample).setImpurity(gbtImpurity).setMaxIter(gbtMaxIter).setMaxDepth(gbtMaxBins).setMaxBins(gbtmaxDepth).setFeatureSubsetStrategy("auto").fit(newtrainingdata)
   val predictionsGBT = gbtModel.transform(validationData)      
   val gbtAUC = new BinaryClassificationEvaluator().setRawPredictionCol("prediction").setLabelCol("label").setMetricName("areaUnderROC").evaluate(predictionsGBT)  
   println("Area Under ROC Curve = " + gbtAUC)
       (( gbtsubsample, gbtImpurity,gbtMaxIter, gbtMaxBins, gbtmaxDepth), gbtModel, gbtAUC)
    }  
    println(gbtGridSearch.sortBy(-_._3).take(5).mkString("\n"))
        val BestModelGBT = gbtGridSearch.sortBy(-_._3).head._2
    BestModelGBT
}   

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
Invalid status code '400' from http://192.168.0.120:8998/sessions/587/statements/37 with error payload: {"msg":"requirement failed: Session isn't active."}
